In [ ]:
!pip install pyswarms

In [17]:
import numpy as np
import pyswarms as ps
from pyswarms.utils.functions import single_obj as fx
from scipy.optimize import fsolve
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from keras import losses

# Primjer 1

In [18]:
max_bound = [5.12, 5.12]
min_bound = [-5.12, -5.12]
bounds = (min_bound, max_bound)
options = {"c1": 0.5, "c2": 0.3, "w": 0.9}
optimizer = ps.single.GlobalBestPSO(n_particles=10, dimensions=2, bounds=bounds, options=options)
optimizer.optimize(fx.rastrigin, iters=1000)

2025-06-11 19:05:33,390 - pyswarms.single.global_best - INFO - Optimize for 1000 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|1000/1000, best_cost=0
2025-06-11 19:05:34,956 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.0, best pos: [ 3.17038560e-09 -1.09711125e-10]


(np.float64(0.0), array([ 3.17038560e-09, -1.09711125e-10]))

# Primjer 2

In [19]:
def cost_function(I):
    U = 10
    R = 100
    I_s = 9.4e-12
    V_t = 25.85e-3

    c = abs(U - V_t * np.log(abs(I[:, 0] / I_s)) - R * I[:, 0])

    return c

In [20]:
options = {"c1": 0.5, "c2": 0.3, "w": 0.9}
optimizer = ps.single.GlobalBestPSO(n_particles=10, dimensions=1, options=options)
cost, pos = optimizer.optimize(cost_function, iters=200)

2025-06-11 19:05:34,982 - pyswarms.single.global_best - INFO - Optimize for 200 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|200/200, best_cost=4.32e-5
2025-06-11 19:05:35,406 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 4.3164137906614997e-05, best pos: [0.09404726]


In [21]:
def cost(I):
    return abs(10-25.85e-3 * np.log(abs(I / 9.4e-12)) - 100 * I)

In [22]:
initial_guess = 0.09
I = fsolve(func=cost, x0=initial_guess)
print(I[0])

0.09404768643017938


# Zadatak 1

In [23]:
data = load_iris()
X = data.data
y = data.target

y_new = np.zeros([X.shape[0], 3])

for i in range(X.shape[0]):
    y_new[i, y[i]] = 1.

y = y_new

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [24]:
n_inputs = 4
n_hidden = 20
n_classes = 3

num_samples = 150

def probs_function(p, X):
    W1 = p[0:80].reshape((n_inputs, n_hidden))
    b1 = p[80:100].reshape((n_hidden,))
    W2 = p[100:160].reshape((n_hidden, n_classes))
    b2 = p[160:163].reshape((n_classes,))

    z1 = X.dot(W1) + b1
    a1 = np.tanh(z1)
    logits = a1.dot(W2) + b2

    exp_scores = np.exp(logits)
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

    return probs

In [25]:
def forward_prop(params):
    probs = probs_function(params, X_train)
    cce = losses.CategoricalCrossentropy()
    loss = cce(y_train, probs).numpy()
    return loss

In [26]:
def f(x):
    n_particles = x.shape[0]
    values = [forward_prop(x[i]) for i in range(n_particles)]
    return np.array(values)

In [27]:
%%time
options = {"c1": 0.5, "c2": 0.3, "w": 0.9}
dimensions = (n_inputs * n_hidden) + (n_hidden * n_classes) + n_hidden + n_classes
optimizer = ps.single.GlobalBestPSO(n_particles=100, dimensions=dimensions, options=options)
cost, pos = optimizer.optimize(f, iters=100)

2025-06-11 19:05:35,521 - pyswarms.single.global_best - INFO - Optimize for 100 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best: 100%|██████████|100/100, best_cost=0.0595
2025-06-11 19:06:15,613 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 0.0594724602997303, best pos: [-2.37194170e-01  2.25778907e-01  1.93580753e-01 -4.67565350e-02
  1.84541804e+00  1.34038132e+00 -6.94635152e-01  8.28599313e-01
 -1.78883807e-01 -2.26140808e-01  7.75556633e-01 -2.90099871e+00
  1.41291234e+00 -1.18851586e+00 -3.25381427e-02  1.36848936e-01
  1.06903937e+00 -8.05899315e-01  8.19919088e-01  6.57690733e-01
 -8.51921421e-01  5.35277924e-01 -6.38993774e-01  7.83891361e-01
 -5.81856268e-01 -2.41346955e+00  1.30768390e+00  1.09423206e+00
  4.69918734e-01  1.35891472e+00 -9.43555073e-03 -1.59980160e+00
  4.77113575e-01 -1.01009581e+00 -6.54242461e-01  4.46537921e-01
 -2.12902262e-01  5.13153512e-01  4.45018796e-01  2.52550396e-01
  7.49875128e-01 -3.2887984

CPU times: user 34.6 s, sys: 1.61 s, total: 36.2 s
Wall time: 40.1 s


In [28]:
def predict(pos):
    probs = probs_function(pos, X_test)
    return probs

In [29]:
(np.round(predict(pos)).astype("int") == y_test).mean()

np.float64(1.0)